# V4: confirm the selected learning rate across all participants
Freeze raw input and LR=0.0001 after the preceding four-case comparison. Evaluate all 12 participants at seeds 45 and 62. Four completed fits are reused, leaving 20 new fits (roughly 45–60 minutes at the previous GPU speed).
Whole-participant validation, CNN, epoch limit, and macro-F1 checkpointing stay the same. This extends participant coverage; it remains exploratory because earlier results informed the learning-rate choice. We are not adding another learning-rate grid.

## 1. Setup
Select a GPU runtime. Code is fixed at tag `v4-low-lr-confirm-v1`.

In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v4-confirm'
run_ref = 'v4-low-lr-confirm-v1'
if not os.path.isdir(repo):
    subprocess.run(['git', 'clone', '-q', '--branch', run_ref, '--single-branch',
                    'https://github.com/parmisbathayan/reproduce-teco-results.git', repo], check=True)
os.chdir(repo)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == subprocess.check_output(['git', 'rev-parse', run_ref + '^{commit}'], text=True).strip()
assert not subprocess.check_output(['git', 'status', '--porcelain'], text=True).strip(), 'Use a clean checkout of the fixed revision.'
print('Ready. Git commit:', commit, flush=True)


## 2. Reuse four results and run the remaining 20
Code, raw data, runtime, and training settings are checked before reuse. Each fit prints START, then SAVED. Completed fits resume after a disconnect.

In [ ]:
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
out_dir = os.path.join(base, 'v4_low_lr_confirmation_45_62')
command = [sys.executable, '-u', '-B', 'src/v4_low_lr_confirmation.py',
           '--trt-dir', '/content/drive/MyDrive/Thesis/Data/TRT_Total',
           '--labels-csv', '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
           '--source-out-dir', os.path.join(base, 'v4_lr_screen_JAY_ARM_45_62'),
           '--out-dir', out_dir, '--config', 'configs/v4_low_lr_confirmation.json']
print('Starting fixed-rate confirmation. Live output follows:', flush=True)
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print('Stopped. Completed fits remain saved.', flush=True)
        raise
if returncode:
    raise subprocess.CalledProcessError(returncode, command)


## 3. Display results
The target is 24 saved fits: 12 participants × 2 seeds. Four are imported from the previous diagnostic and marked in fit_origin.

In [ ]:
import pandas as pd
from IPython.display import display
metrics = pd.read_csv(os.path.join(out_dir, 'fold_metrics.csv'))
print('Fits saved:', len(metrics), '/ 24')
display(pd.read_csv(os.path.join(out_dir, 'summary_by_seed.csv')))
display(metrics[['seed', 'participant', 'macro_f1', 'collapsed', 'fit_origin']])
